# Latent diffusion & text-to-image — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## diffusion in learned latent space
Autoencoders (10.1), VAEs (10.2), DDPMs (10.12), and classifier-free guidance (10.13) combine here. Text encoders provide conditioning, while the decoder turns latent samples back into images. ControlNet (10.15) adds more structured conditioning on top.

In [ ]:
# 1) decode a two-dimensional latent vector
z=np.array([1.,-1.]); D=np.array([[1.,.5],[.2,1.]]); x=D@z
plt.figure(figsize=(4,4)); plt.quiver([0],[0],[x[0]],[x[1]],angles='xy',scale_units='xy',scale=1); plt.xlim(-1,1); plt.ylim(-1.2,.5); plt.title('latent decoded to pixel-like space')
assert np.allclose(x,[0.5,-0.8])

In [ ]:
# 2) latent diffusion works on fewer coordinates than pixels
latent_dims=np.array([2,4,8]); pixel_dims=np.array([16,64,256])
plt.figure(figsize=(5,3)); plt.plot(latent_dims,label='latent dims'); plt.plot(pixel_dims,label='pixel dims'); plt.legend(); plt.title('compression reduces denoising size')
assert pixel_dims[-1]>latent_dims[-1]

In [ ]:
# 3) noising in latent space follows DDPM algebra
abar=.64; eps=np.array([-.5,.25]); zt=math.sqrt(abar)*z+math.sqrt(1-abar)*eps
plt.figure(figsize=(4,4)); plt.scatter([z[0]],[z[1]],label='z0'); plt.scatter([zt[0]],[zt[1]],label='zt'); plt.legend(); plt.title('latent noising')
assert zt.shape==(2,)

In [ ]:
# 4) text conditioning shifts the latent denoising direction
uncond=np.array([-.2,.1]); text=np.array([.6,.4]); guided=uncond+2*(text-uncond)
plt.figure(figsize=(4,4)); plt.quiver([0,0],[0,0],[uncond[0],guided[0]],[uncond[1],guided[1]],angles='xy',scale_units='xy',scale=1); plt.title('text-guided latent update')
assert np.linalg.norm(guided)>np.linalg.norm(uncond)

In [ ]:
# 5) decoder errors cap output quality
ztrue=np.array([1.,-1.]); zhat=ztrue+np.array([.1,-.2]); err=np.linalg.norm(D@ztrue-D@zhat)
plt.figure(figsize=(4,3)); plt.bar(['latent err','decoded err'],[np.linalg.norm(ztrue-zhat),err]); plt.title('latent error propagates through decoder')
assert err>0